# Completed Thesis Analysis Verification Notebook

This notebook verifies the cleaned data, DEA efficiency computation, descriptive tables, and second-stage regression for the Estose thesis. It uses the cleaned Excel files in `outputs/cleaned_data/` and exports thesis tables and figures to `outputs/completed_thesis/`.

In [ ]:
from pathlib import Path
import sys
print(sys.executable)
print(Path.cwd())
%matplotlib inline

In [ ]:
from __future__ import annotations

import json
import textwrap
from pathlib import Path

import nbformat as nbf
import numpy as np
import pandas as pd
from scipy.optimize import linprog, minimize
from scipy.stats import norm
import statsmodels.api as sm

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns


ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_FILE = ROOT / "outputs" / "cleaned_data" / "ESTOSE-data1-cleaned.xlsx"
DMU_FILE = ROOT / "outputs" / "cleaned_data" / "Updated-DMU1-cleaned.xlsx"
OUT_DIR = ROOT / "outputs" / "completed_thesis"
FIG_DIR = OUT_DIR / "figures"
NOTEBOOK = ROOT / "notebooks" / "estose_completed_thesis_analysis.ipynb"

INPUT_COLS = [
    "land_ha",
    "bearing_trees",
    "organic_fertilizer_kg_tree_year",
    "inorganic_fertilizer_kg_tree_year",
    "chemicals_liter_tree_year",
    "labor_man_days_tree_year",
]
OUTPUT_COLS = ["output_kg_tree_year"]

EDUCATION_YEARS = {
    "Elementary Undergraduate": 3,
    "ALS": 6,
    "High School Undergraduate": 8,
    "High School Graduate": 10,
    "College Undergraduate": 12,
    "College Graduate": 14,
    "Master's Degree": 16,
    "Doctoral Degree": 18,
}


def ensure_dirs() -> None:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    NOTEBOOK.parent.mkdir(parents=True, exist_ok=True)


def load_data() -> tuple[dict[str, pd.DataFrame], pd.DataFrame]:
    sheets = pd.read_excel(DATA_FILE, sheet_name=None)
    dmu = pd.read_excel(DMU_FILE)
    return sheets, dmu


def prepare_dmu(dmu: pd.DataFrame) -> pd.DataFrame:
    df = dmu.copy()
    df = df.dropna(how="all").dropna(subset=["DMU"]).copy()
    df["DMU"] = df["DMU"].astype(int)
    df["anonymous_id"] = df["DMU"].map(lambda x: f"DMU-{x:02d}")
    df["land_ha"] = df["LAND (sq.m)"].astype(float) / 10000
    df["bearing_trees"] = df["NO. OF BEARING TREES"].astype(float)
    df["output_kg_tree_year"] = df["OUTPUT (kg/tree/year)"].astype(float)
    df["organic_fertilizer_kg_tree_year"] = df["ORGANIC FERTILIZATION (kg/tree/year)"].astype(float)
    df["inorganic_fertilizer_kg_tree_year"] = df["INORGANIC FERTILIZATION (kg/tree/year)"].astype(float)
    df["chemicals_liter_tree_year"] = df["CHEMICALS (Liter/tree/year)"].astype(float)
    df["labor_man_days_tree_year"] = df["LABOR (man-days/tree/year)"].astype(float)
    df["total_output_kg_year"] = df["output_kg_tree_year"] * df["bearing_trees"]
    return df


def validate_data(sheets: dict[str, pd.DataFrame], dmu: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    issues: list[dict[str, object]] = []
    expected_ids = set(range(1, 31))
    for sheet_name, df in sheets.items():
        if "respondent_id" not in df.columns:
            issues.append({"area": sheet_name, "issue": "Missing respondent_id column", "severity": "high"})
            continue
        ids = set(pd.to_numeric(df["respondent_id"], errors="coerce").dropna().astype(int))
        if ids != expected_ids:
            issues.append(
                {
                    "area": sheet_name,
                    "issue": f"ID coverage mismatch; missing={sorted(expected_ids - ids)}, extra={sorted(ids - expected_ids)}",
                    "severity": "high",
                }
            )
    if len(dmu) != 30 or dmu["DMU"].nunique() != 30:
        issues.append({"area": "DMU", "issue": "DMU file must contain 30 unique DMUs", "severity": "high"})
    for col in INPUT_COLS + OUTPUT_COLS:
        bad = dmu[col].isna() | (dmu[col] <= 0)
        if bad.any():
            issues.append({"area": "DMU", "issue": f"{col} has non-positive or missing values", "severity": "high"})

    farm = sheets["Farm Characteristics"]
    out = sheets["Production Outputs"]
    merged = farm.merge(out, on="respondent_id", how="inner")
    merged["expected_land_ha"] = merged["farm_area_ha"].astype(float)
    merged["expected_bearing_trees"] = merged["no_bearing_jackfruit_trees"].astype(float)
    merged["expected_output"] = merged["avg_yield_bearingtree"].astype(float) * merged["avg_weight_fruit_kg"].astype(float)
    comp = merged.merge(dmu.rename(columns={"DMU": "respondent_id"}), on="respondent_id", how="inner")
    match_checks = pd.DataFrame(
        {
            "respondent_id": comp["respondent_id"],
            "land_ha_diff": comp["expected_land_ha"] - comp["land_ha"],
            "bearing_tree_diff": comp["expected_bearing_trees"] - comp["bearing_trees"],
            "output_diff": comp["expected_output"] - comp["output_kg_tree_year"],
        }
    )
    for col in ["land_ha_diff", "bearing_tree_diff", "output_diff"]:
        if (match_checks[col].abs() > 1e-8).any():
            issues.append({"area": "DMU merge", "issue": f"{col} mismatch remains", "severity": "high"})

    reg = build_regression_frame(sheets, dmu, with_scores=False)
    reg_cols = [
        "age",
        "education_years",
        "monthly_income",
        "years_jackfruit_farming",
        "farm_area_ha",
        "technology_adoption",
        "recommended_fertilizer_rate",
        "farm_records",
    ]
    for col in reg_cols:
        if reg[col].isna().any():
            issues.append({"area": "Regression", "issue": f"{col} has missing values", "severity": "medium"})

    return pd.DataFrame(issues), match_checks


def dea_input_oriented(df: pd.DataFrame, returns: str) -> tuple[pd.Series, pd.DataFrame]:
    x = df[INPUT_COLS].to_numpy(dtype=float)
    y = df[OUTPUT_COLS].to_numpy(dtype=float)
    n, m = x.shape
    scores: list[float] = []
    lambdas: list[np.ndarray] = []
    for o in range(n):
        c = np.zeros(n + 1)
        c[-1] = 1.0
        a_ub, b_ub = [], []
        for i in range(m):
            row = np.zeros(n + 1)
            row[:n] = x[:, i]
            row[-1] = -x[o, i]
            a_ub.append(row)
            b_ub.append(0.0)
        row = np.zeros(n + 1)
        row[:n] = -y[:, 0]
        a_ub.append(row)
        b_ub.append(-y[o, 0])
        a_eq, b_eq = None, None
        if returns == "vrs":
            a_eq = np.zeros((1, n + 1))
            a_eq[0, :n] = 1
            b_eq = np.array([1.0])
        elif returns == "nirs":
            row = np.zeros(n + 1)
            row[:n] = 1
            a_ub.append(row)
            b_ub.append(1.0)
        elif returns != "crs":
            raise ValueError("returns must be crs, vrs, or nirs")
        result = linprog(
            c,
            A_ub=np.array(a_ub),
            b_ub=np.array(b_ub),
            A_eq=a_eq,
            b_eq=b_eq,
            bounds=[(0, None)] * n + [(0, 1)],
            method="highs",
        )
        if not result.success:
            raise RuntimeError(f"{returns.upper()} DEA failed for DMU {df.loc[o, 'DMU']}: {result.message}")
        scores.append(float(np.clip(result.x[-1], 0, 1)))
        lambdas.append(result.x[:n])
    lambda_df = pd.DataFrame(lambdas, columns=[f"peer_DMU_{int(x):02d}" for x in df["DMU"]])
    lambda_df.insert(0, "DMU", df["DMU"].values)
    return pd.Series(scores, index=df.index, name=f"{returns}_te"), lambda_df


def classify_rts(crs: float, vrs: float, nirs: float) -> str:
    if abs(crs - vrs) <= 1e-6:
        return "CRS"
    if abs(nirs - vrs) <= 1e-6:
        return "DRS"
    return "IRS"


def compute_dea(dmu: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    results = dmu.copy()
    results["crs_te"], _ = dea_input_oriented(results, "crs")
    results["vrs_te"], lambda_vrs = dea_input_oriented(results, "vrs")
    results["nirs_te"], _ = dea_input_oriented(results, "nirs")
    results["scale_efficiency"] = (results["crs_te"] / results["vrs_te"]).clip(upper=1)
    results["returns_to_scale"] = [
        classify_rts(c, v, n) for c, v, n in zip(results["crs_te"], results["vrs_te"], results["nirs_te"])
    ]
    results["vrs_status"] = np.where(results["vrs_te"] >= 0.999999, "Technically efficient", "Technically inefficient")
    results["crs_status"] = np.where(results["crs_te"] >= 0.999999, "Technically efficient", "Technically inefficient")
    results["potential_input_reduction_pct"] = (1 - results["vrs_te"]) * 100
    return results, lambda_vrs


def yes_no(series: pd.Series) -> pd.Series:
    return series.map({"Yes": 1, "No": 0}).astype(float)


def build_regression_frame(
    sheets: dict[str, pd.DataFrame], dmu_or_results: pd.DataFrame, with_scores: bool = True
) -> pd.DataFrame:
    socio = sheets["Socioeconomic Profile"].copy()
    farm = sheets["Farm Characteristics"].copy()
    support = sheets["Farmer Support"].copy()
    practice = sheets["Farming Practice and Management"].copy()
    reg = socio.merge(farm, on="respondent_id").merge(support, on="respondent_id").merge(practice, on="respondent_id")
    reg["education_years"] = reg["education"].map(EDUCATION_YEARS)
    reg["log_monthly_income"] = np.log(reg["monthly_income"].astype(float))
    reg["training_attended"] = yes_no(reg["attended_training"])
    reg["credit_access"] = yes_no(reg["access_credit_sources"])
    reg["financial_assistance"] = yes_no(reg["receive_financial_assitance"])
    reg["technology_adoption"] = yes_no(reg["adopted_farming_technologies_practice"])
    reg["recommended_fertilizer_rate"] = yes_no(reg["follow_recommended_fertilizer_rate"])
    reg["farm_records"] = yes_no(reg["farm_records"]).fillna(0)
    if with_scores:
        scores = dmu_or_results[["DMU", "vrs_te", "crs_te", "scale_efficiency", "returns_to_scale"]].rename(
            columns={"DMU": "respondent_id"}
        )
        reg = reg.merge(scores, on="respondent_id", how="inner")
    return reg


def tobit_two_limit(y: np.ndarray, x: np.ndarray, lower: float = 0.0, upper: float = 1.0):
    y = np.asarray(y, dtype=float)
    x = np.asarray(x, dtype=float)
    n, k = x.shape
    beta0 = np.linalg.lstsq(x, np.clip(y, lower + 1e-5, upper - 1e-5), rcond=None)[0]
    resid = y - x @ beta0
    sigma0 = max(float(np.std(resid)), 0.05)
    start = np.r_[beta0, np.log(sigma0)]

    def nll(params: np.ndarray) -> float:
        beta = params[:k]
        sigma = np.exp(params[k])
        mu = x @ beta
        lower_mask = y <= lower + 1e-9
        upper_mask = y >= upper - 1e-9
        mid_mask = ~(lower_mask | upper_mask)
        ll = np.zeros(n)
        ll[lower_mask] = norm.logcdf((lower - mu[lower_mask]) / sigma)
        ll[upper_mask] = norm.logsf((upper - mu[upper_mask]) / sigma)
        ll[mid_mask] = norm.logpdf((y[mid_mask] - mu[mid_mask]) / sigma) - np.log(sigma)
        return -np.sum(ll)

    result = minimize(nll, start, method="BFGS")
    hess_inv = np.asarray(result.hess_inv) if hasattr(result, "hess_inv") else np.full((k + 1, k + 1), np.nan)
    se = np.sqrt(np.diag(hess_inv)) if hess_inv.shape == (k + 1, k + 1) else np.full(k + 1, np.nan)
    params = result.x
    z = params / se
    p = 2 * (1 - norm.cdf(np.abs(z)))
    return result, params, se, z, p


def run_regression(reg: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, dict[str, object]]:
    # Defensible model: exclude ultra-sparse credit/training dummies from primary regression.
    predictors = [
        "age",
        "education_years",
        "log_monthly_income",
        "years_jackfruit_farming",
        "farm_area_ha",
    ]
    model = reg[["respondent_id", "vrs_te"] + predictors].dropna().copy()
    y = model["vrs_te"].to_numpy(float)
    x_raw = model[predictors].astype(float)
    x_std = (x_raw - x_raw.mean()) / x_raw.std(ddof=0)
    x_std = x_std.replace([np.inf, -np.inf], 0).fillna(0)
    X = np.column_stack([np.ones(len(x_std)), x_std.to_numpy(float)])
    names = ["Constant"] + predictors

    tob_result, params, se, z, p = tobit_two_limit(y, X, 0, 1)
    tobit_table = pd.DataFrame(
        {
            "Variable": names + ["Sigma"],
            "Coefficient": params,
            "Std. Error": se,
            "z": z,
            "p-value": p,
        }
    )
    tobit_table.loc[tobit_table["Variable"] == "Sigma", "Coefficient"] = np.exp(params[-1])
    tobit_table.loc[tobit_table["Variable"] == "Sigma", ["z", "p-value"]] = np.nan

    ols = sm.OLS(y, X).fit(cov_type="HC3")
    ols_table = pd.DataFrame(
        {
            "Variable": names,
            "Coefficient": ols.params,
            "Robust Std. Error": ols.bse,
            "t": ols.tvalues,
            "p-value": ols.pvalues,
        }
    )
    ols_diagnostics = pd.DataFrame(
        {
            "DMU": model["respondent_id"].astype(int).map(lambda x: f"DMU-{x:02d}"),
            "Observed VRS TE": y,
            "OLS Fitted VRS TE": ols.fittedvalues,
            "OLS Residual": ols.resid,
        }
    )
    predictor_correlation = x_raw.corr().reset_index().rename(columns={"index": "Variable"})
    diagnostics = {
        "n": int(len(model)),
        "upper_censored_count": int((model["vrs_te"] >= 0.999999).sum()),
        "lower_censored_count": int((model["vrs_te"] <= 1e-9).sum()),
        "tobit_converged": bool(tob_result.success),
        "tobit_message": str(tob_result.message),
        "ols_r_squared": float(ols.rsquared),
        "ols_adj_r_squared": float(ols.rsquared_adj),
        "primary_predictors": predictors,
        "excluded_sparse_predictors": {
            "training_attended": reg["training_attended"].value_counts(dropna=False).to_dict(),
            "credit_access": reg["credit_access"].value_counts(dropna=False).to_dict(),
            "financial_assistance": reg["financial_assistance"].value_counts(dropna=False).to_dict(),
            "technology_adoption": reg["technology_adoption"].value_counts(dropna=False).to_dict(),
            "recommended_fertilizer_rate": reg["recommended_fertilizer_rate"].value_counts(dropna=False).to_dict(),
            "farm_records": reg["farm_records"].value_counts(dropna=False).to_dict(),
        },
    }
    return tobit_table, ols_table, ols_diagnostics, predictor_correlation, diagnostics


def summarize_tables(sheets: dict[str, pd.DataFrame], dmu_results: pd.DataFrame, reg: pd.DataFrame) -> dict[str, pd.DataFrame]:
    socio = sheets["Socioeconomic Profile"]
    farm = sheets["Farm Characteristics"].copy()
    out = sheets["Production Outputs"]
    market = sheets["Marketing Practices"]
    support = sheets["Farmer Support"]
    challenges = sheets["Challenges"]

    farm["production_system"] = farm["production_system"].replace({"Mnocropping": "Monocropping"})

    socio_numeric = socio[["age", "household_size", "monthly_income", "years_jackfruit_farming"]].describe().T
    socio_numeric = socio_numeric[["count", "mean", "std", "min", "50%", "max"]].reset_index().rename(columns={"index": "Variable"})
    farm_numeric = farm[
        ["farm_area_ha", "no_bearing_jackfruit_trees", "no_nonbearing_jackfruit_trees", "tree_age_bearing"]
    ].describe().T
    farm_numeric = farm_numeric[["count", "mean", "std", "min", "50%", "max"]].reset_index().rename(columns={"index": "Variable"})
    output_numeric = out[["avg_yield_bearingtree", "avg_weight_fruit_kg", "no_fruits_harvested_year", "farmgate_price"]].describe().T
    output_numeric = output_numeric[["count", "mean", "std", "min", "50%", "max"]].reset_index().rename(columns={"index": "Variable"})

    def freq(df: pd.DataFrame, col: str, label: str) -> pd.DataFrame:
        vc = df[col].value_counts(dropna=False).reset_index()
        vc.columns = ["Category", "Frequency"]
        vc["Percent"] = vc["Frequency"] / len(df) * 100
        vc.insert(0, "Variable", label)
        return vc

    profile_freq = pd.concat(
        [
            freq(socio, "sex", "Sex"),
            freq(socio, "education", "Educational attainment"),
            freq(socio, "primary_income_source", "Main source of income"),
            freq(farm, "production_system", "Production system"),
            freq(support, "attended_training", "Training attendance"),
            freq(support, "receive_technical_advice", "Technical advice"),
            freq(support, "access_credit_sources", "Credit access"),
            freq(support, "adopted_farming_technologies_practice", "Technology adoption"),
        ],
        ignore_index=True,
    )

    org_clean = socio["org_membership"].fillna("No reported organization").astype(str).str.strip()
    org_category = np.where(
        org_clean.str.contains(",", regex=False),
        "Multiple organizational memberships",
        np.where(
            org_clean.eq("Baybay Jackfruit Growers' Association"),
            "Baybay Jackfruit Growers' Association only",
            org_clean,
        ),
    )
    org_membership = pd.Series(org_category, name="Category").value_counts().reset_index()
    org_membership.columns = ["Category", "Frequency"]
    org_membership["Percent"] = org_membership["Frequency"] / len(socio) * 100

    variable_specs = [
        ("Input", "Land area", "hectares", "land_ha"),
        ("Input", "Number of bearing trees", "trees", "bearing_trees"),
        ("Input", "Organic fertilizer", "kg/tree/year", "organic_fertilizer_kg_tree_year"),
        ("Input", "Inorganic fertilizer", "kg/tree/year", "inorganic_fertilizer_kg_tree_year"),
        ("Input", "Chemicals", "liter/tree/year", "chemicals_liter_tree_year"),
        ("Input", "Labor", "man-days/tree/year", "labor_man_days_tree_year"),
        ("Output", "Jackfruit output", "kg/tree/year", "output_kg_tree_year"),
    ]
    dea_input_output_summary = pd.DataFrame(
        [
            {
                "Role": role,
                "Variable": label,
                "Unit": unit,
                "count": dmu_results[col].count(),
                "mean": dmu_results[col].mean(),
                "std": dmu_results[col].std(),
                "min": dmu_results[col].min(),
                "50%": dmu_results[col].median(),
                "max": dmu_results[col].max(),
            }
            for role, label, unit, col in variable_specs
        ]
    )

    dea_summary = dmu_results[["crs_te", "vrs_te", "scale_efficiency", "potential_input_reduction_pct"]].describe().T
    dea_summary = dea_summary[["count", "mean", "std", "min", "50%", "max"]].reset_index().rename(columns={"index": "Measure"})
    tech_table = dmu_results[
        ["anonymous_id", "crs_te", "vrs_te", "crs_status", "vrs_status", "potential_input_reduction_pct"]
    ].rename(
        columns={
            "anonymous_id": "DMU",
            "crs_te": "CRS TE",
            "vrs_te": "VRS TE",
            "crs_status": "CRS Status",
            "vrs_status": "VRS Status",
            "potential_input_reduction_pct": "Potential Input Reduction (%)",
        }
    )
    scale_table = dmu_results[["anonymous_id", "crs_te", "vrs_te", "scale_efficiency", "returns_to_scale"]].rename(
        columns={
            "anonymous_id": "DMU",
            "crs_te": "CRS TE",
            "vrs_te": "VRS TE",
            "scale_efficiency": "Scale Efficiency",
            "returns_to_scale": "Returns to Scale",
        }
    )
    rts_count = dmu_results["returns_to_scale"].value_counts().rename_axis("Returns to Scale").reset_index(name="Frequency")
    rts_count["Percent"] = rts_count["Frequency"] / len(dmu_results) * 100

    challenge_cols = [c for c in challenges.columns if c not in ["respondent_id", "E1", "E2"]]
    challenge_map = {
        "A1": "High input cost",
        "A2": "Limited capital/credit",
        "A3": "Labor shortage",
        "A4": "Limited tools/equipment",
        "B1": "Lack of technical training",
        "B2": "Limited pest/disease knowledge",
        "B3": "Difficulty applying recommended practices",
        "B4": "Limited extension access",
        "C1": "Climate variability",
        "C2": "Pest and disease outbreaks",
        "C3": "Poor soil condition",
        "D1": "High postharvest losses",
        "D2": "Lack of storage facilities",
        "D3": "Price fluctuation",
        "D4": "Limited stable buyers",
    }
    challenge_summary = (
        challenges[challenge_cols]
        .mean()
        .rename_axis("Code")
        .reset_index(name="Mean Score")
        .assign(Challenge=lambda x: x["Code"].map(challenge_map))
        .sort_values("Mean Score", ascending=False)[["Code", "Challenge", "Mean Score"]]
    )

    rts_counts = dmu_results["returns_to_scale"].value_counts().to_dict()
    top_challenge = challenge_summary.iloc[0]
    second_challenge = challenge_summary.iloc[1]
    recommendation_action_matrix = pd.DataFrame(
        [
            {
                "Basis from Results": f"{int((dmu_results['vrs_te'] >= 0.999999).sum())} DMUs were VRS-efficient.",
                "Interpretation": "Most farms performed well after allowing for farm-scale differences.",
                "Recommended Action": "Use VRS-efficient farms as local benchmarks for input timing, pruning, fertilization, pest management, and labor organization.",
            },
            {
                "Basis from Results": f"Mean scale efficiency was {dmu_results['scale_efficiency'].mean():.3f}.",
                "Interpretation": "Scale conditions still explain part of the remaining inefficiency.",
                "Recommended Action": "Use CRS, IRS, and DRS classifications when designing farm-specific technical assistance instead of applying one uniform intervention.",
            },
            {
                "Basis from Results": f"{rts_counts.get('IRS', 0)} DMUs operated under IRS.",
                "Interpretation": "These farms may gain from better scale support if market and agronomic conditions permit.",
                "Recommended Action": "Assess expansion readiness, input access, market linkage, and bearing-tree management before recommending scale increases.",
            },
            {
                "Basis from Results": f"{rts_counts.get('DRS', 0)} DMUs operated under DRS.",
                "Interpretation": "These farms may be using inputs or operating at a scale that does not proportionally increase output.",
                "Recommended Action": "Review input intensity, tree productivity, labor deployment, and farm organization before adding more inputs.",
            },
            {
                "Basis from Results": f"Top challenges were {top_challenge['Challenge']} and {second_challenge['Challenge']}.",
                "Interpretation": "Remaining inefficiency is linked not only to input quantity but also to production risk and management constraints.",
                "Recommended Action": "Prioritize climate-resilient production support, pest and disease management, and practical field-level training.",
            },
            {
                "Basis from Results": "Regression had 27 upper-censored VRS TE scores.",
                "Interpretation": "The determinants model has limited statistical power in the current 30-farm dataset.",
                "Recommended Action": "Avoid overstating causal determinants; expand sample size and collect multiple-year data in future studies.",
            },
        ]
    )

    return {
        "socio_numeric": socio_numeric,
        "farm_numeric": farm_numeric,
        "output_numeric": output_numeric,
        "profile_frequencies": profile_freq,
        "organization_membership": org_membership,
        "dea_input_output_summary": dea_input_output_summary,
        "dea_summary": dea_summary,
        "technical_efficiency_table": tech_table,
        "scale_efficiency_table": scale_table,
        "returns_to_scale_count": rts_count,
        "challenge_summary": challenge_summary,
        "recommendation_action_matrix": recommendation_action_matrix,
    }


def save_tables(
    tables: dict[str, pd.DataFrame],
    dmu_results: pd.DataFrame,
    reg: pd.DataFrame,
    validation: pd.DataFrame,
    match: pd.DataFrame,
    tobit: pd.DataFrame,
    ols: pd.DataFrame,
    ols_diagnostics: pd.DataFrame,
    predictor_correlation: pd.DataFrame,
    diagnostics: dict[str, object],
) -> None:
    for name, table in tables.items():
        table.to_csv(OUT_DIR / f"{name}.csv", index=False)
    dmu_results.to_csv(OUT_DIR / "dea_full_results.csv", index=False)
    reg.to_csv(OUT_DIR / "regression_dataset.csv", index=False)
    validation.to_csv(OUT_DIR / "validation_issues.csv", index=False)
    match.to_csv(OUT_DIR / "validation_dmu_match.csv", index=False)
    tobit.to_csv(OUT_DIR / "tobit_regression_results.csv", index=False)
    ols.to_csv(OUT_DIR / "ols_robustness_results.csv", index=False)
    ols_diagnostics.to_csv(OUT_DIR / "ols_observed_fitted_residuals.csv", index=False)
    predictor_correlation.to_csv(OUT_DIR / "predictor_correlation_matrix.csv", index=False)
    (OUT_DIR / "regression_diagnostics.json").write_text(json.dumps(diagnostics, indent=2))


def save_figures(
    tables: dict[str, pd.DataFrame],
    dmu_results: pd.DataFrame,
    reg: pd.DataFrame,
    tobit: pd.DataFrame,
    ols_diagnostics: pd.DataFrame,
    predictor_correlation: pd.DataFrame,
) -> None:
    sns.set_theme(style="whitegrid", context="notebook")

    fig, ax = plt.subplots(figsize=(8, 5))
    sns.histplot(dmu_results["vrs_te"], bins=8, kde=True, color="#4C78A8", ax=ax)
    ax.set_title("Distribution of VRS Technical Efficiency Scores")
    ax.set_xlabel("VRS Technical Efficiency")
    ax.set_ylabel("Number of DMUs")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "fig_vrs_distribution.png", dpi=220)
    plt.close(fig)

    sorted_vrs = dmu_results.sort_values("vrs_te")
    fig, ax = plt.subplots(figsize=(9, 7))
    ax.barh(sorted_vrs["anonymous_id"], sorted_vrs["vrs_te"], color=np.where(sorted_vrs["vrs_te"] >= 0.999999, "#2F855A", "#C05621"))
    ax.axvline(1, color="black", linewidth=1)
    ax.set_title("VRS Technical Efficiency by DMU")
    ax.set_xlabel("VRS Technical Efficiency")
    ax.set_ylabel("DMU")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "fig_vrs_by_dmu.png", dpi=220)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(6.5, 6))
    sns.scatterplot(data=dmu_results, x="crs_te", y="vrs_te", hue="returns_to_scale", s=90, ax=ax)
    ax.plot([0, 1], [0, 1], linestyle="--", color="black", linewidth=1)
    ax.set_title("CRS and VRS Technical Efficiency")
    ax.set_xlabel("CRS Technical Efficiency")
    ax.set_ylabel("VRS Technical Efficiency")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "fig_crs_vs_vrs.png", dpi=220)
    plt.close(fig)

    sorted_se = dmu_results.sort_values("scale_efficiency")
    fig, ax = plt.subplots(figsize=(9, 7))
    ax.barh(sorted_se["anonymous_id"], sorted_se["scale_efficiency"], color="#4C78A8")
    ax.axvline(1, color="black", linewidth=1)
    ax.set_title("Scale Efficiency by DMU")
    ax.set_xlabel("Scale Efficiency")
    ax.set_ylabel("DMU")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "fig_scale_efficiency.png", dpi=220)
    plt.close(fig)

    rts = tables["returns_to_scale_count"]
    fig, ax = plt.subplots(figsize=(6, 4.5))
    ax.bar(rts["Returns to Scale"], rts["Frequency"], color="#4C78A8")
    ax.set_title("Returns-to-Scale Classification")
    ax.set_xlabel("Returns to Scale")
    ax.set_ylabel("Number of DMUs")
    for i, val in enumerate(rts["Frequency"]):
        ax.text(i, val + 0.2, str(int(val)), ha="center")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "fig_returns_to_scale.png", dpi=220)
    plt.close(fig)

    challenge = tables["challenge_summary"].head(10).sort_values("Mean Score")
    fig, ax = plt.subplots(figsize=(8.5, 6))
    ax.barh(challenge["Challenge"], challenge["Mean Score"], color="#4C78A8")
    ax.set_title("Top Production Efficiency Challenges")
    ax.set_xlabel("Mean Challenge Score (0-4)")
    ax.set_ylabel("")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "fig_top_challenges.png", dpi=220)
    plt.close(fig)

    # Coefficients, excluding constant and sigma.
    coef = tobit[~tobit["Variable"].isin(["Constant", "Sigma"])].copy()
    coef = coef.sort_values("Coefficient")
    fig, ax = plt.subplots(figsize=(8, 5.5))
    ax.barh(coef["Variable"], coef["Coefficient"], color=np.where(coef["Coefficient"] >= 0, "#2F855A", "#C05621"))
    ax.axvline(0, color="black", linewidth=1)
    ax.set_title("Tobit Regression Coefficients for VRS Technical Efficiency")
    ax.set_xlabel("Standardized Coefficient")
    ax.set_ylabel("")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "fig_tobit_coefficients.png", dpi=220)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    ax.scatter(
        ols_diagnostics["Observed VRS TE"],
        ols_diagnostics["OLS Fitted VRS TE"],
        color="#4C78A8",
        edgecolor="black",
        s=75,
        alpha=0.9,
    )
    ax.plot([0, 1], [0, 1], linestyle="--", color="black", linewidth=1)
    ax.set_xlim(0.55, 1.03)
    ax.set_ylim(0.55, 1.03)
    ax.set_title("Observed vs Fitted VRS Technical Efficiency")
    ax.set_xlabel("Observed VRS Technical Efficiency")
    ax.set_ylabel("OLS Fitted VRS Technical Efficiency")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "fig_ols_observed_vs_fitted.png", dpi=220)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(
        ols_diagnostics["OLS Fitted VRS TE"],
        ols_diagnostics["OLS Residual"],
        color="#4C78A8",
        edgecolor="black",
        s=75,
        alpha=0.9,
    )
    ax.axhline(0, color="black", linewidth=1)
    ax.set_title("OLS Residual Plot for VRS Technical Efficiency")
    ax.set_xlabel("OLS Fitted VRS Technical Efficiency")
    ax.set_ylabel("OLS Residual")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "fig_ols_residuals.png", dpi=220)
    plt.close(fig)

    corr = predictor_correlation.set_index("Variable")
    fig, ax = plt.subplots(figsize=(8, 6.5))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, vmin=-1, vmax=1, square=True, ax=ax)
    ax.set_title("Correlation Matrix of Regression Predictors")
    ax.set_xlabel("")
    ax.set_ylabel("")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "fig_predictor_correlation_heatmap.png", dpi=220)
    plt.close(fig)

    efficient_count = int((dmu_results["vrs_te"] >= 0.999999).sum())
    inefficient_count = int((dmu_results["vrs_te"] < 0.999999).sum())
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.hist(dmu_results["vrs_te"], bins=np.linspace(0.55, 1.0, 10), color="#4C78A8", edgecolor="black")
    ax.axvline(1.0, color="#C05621", linewidth=2)
    ax.annotate(
        f"{efficient_count} of {len(dmu_results)} DMUs at 1.000",
        xy=(1.0, efficient_count),
        xytext=(0.79, max(efficient_count * 0.85, inefficient_count + 1)),
        arrowprops={"arrowstyle": "->", "color": "black"},
        ha="left",
    )
    ax.set_title("VRS Technical Efficiency Ceiling Effect")
    ax.set_xlabel("VRS Technical Efficiency")
    ax.set_ylabel("Number of DMUs")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "fig_vrs_ceiling_effect.png", dpi=220)
    plt.close(fig)


def build_summary_json(tables: dict[str, pd.DataFrame], dmu_results: pd.DataFrame, diagnostics: dict[str, object], validation: pd.DataFrame) -> dict[str, object]:
    return {
        "n_respondents": 30,
        "validation_issue_count": int(len(validation)),
        "mean_crs_te": float(dmu_results["crs_te"].mean()),
        "mean_vrs_te": float(dmu_results["vrs_te"].mean()),
        "mean_scale_efficiency": float(dmu_results["scale_efficiency"].mean()),
        "crs_efficient_count": int((dmu_results["crs_te"] >= 0.999999).sum()),
        "vrs_efficient_count": int((dmu_results["vrs_te"] >= 0.999999).sum()),
        "rts_counts": dmu_results["returns_to_scale"].value_counts().to_dict(),
        "lowest_vrs_dmu": str(dmu_results.loc[dmu_results["vrs_te"].idxmin(), "anonymous_id"]),
        "lowest_vrs_score": float(dmu_results["vrs_te"].min()),
        "highest_input_reduction_pct": float(dmu_results["potential_input_reduction_pct"].max()),
        "regression": diagnostics,
    }




In [ ]:
ensure_dirs()
sheets, dmu_raw = load_data()
dmu = prepare_dmu(dmu_raw)
validation, match_checks = validate_data(sheets, dmu)
dmu_results, lambda_vrs = compute_dea(dmu)
reg = build_regression_frame(sheets, dmu_results, with_scores=True)
tobit_table, ols_table, ols_diagnostics, predictor_correlation, diagnostics = run_regression(reg)
tables = summarize_tables(sheets, dmu_results, reg)
save_tables(tables, dmu_results, reg, validation, match_checks, tobit_table, ols_table, ols_diagnostics, predictor_correlation, diagnostics)
save_figures(tables, dmu_results, reg, tobit_table, ols_diagnostics, predictor_correlation)
summary = build_summary_json(tables, dmu_results, diagnostics, validation)
(OUT_DIR / 'analysis_summary.json').write_text(json.dumps(summary, indent=2))
summary

## Data Validation

In [ ]:
display(validation)
display(match_checks.describe().T)

## Descriptive Tables

In [ ]:
display(tables['socio_numeric'])
display(tables['organization_membership'])
display(tables['farm_numeric'])
display(tables['output_numeric'])
display(tables['profile_frequencies'].head(30))

## DEA Results

In [ ]:
display(tables['dea_input_output_summary'])
display(tables['dea_summary'])
display(tables['technical_efficiency_table'])
display(tables['scale_efficiency_table'])
display(tables['returns_to_scale_count'])

## Regression Results

In [ ]:
display(tobit_table)
display(ols_table)
display(ols_diagnostics)
display(predictor_correlation)
diagnostics

## Regression Diagnostic Figures

In [ ]:
from IPython.display import Image, display
for filename in [
    'fig_tobit_coefficients.png',
    'fig_ols_observed_vs_fitted.png',
    'fig_ols_residuals.png',
    'fig_predictor_correlation_heatmap.png',
    'fig_vrs_ceiling_effect.png',
]:
    display(Image(filename=str(FIG_DIR / filename)))

## Recommendation Matrix

In [ ]:
display(tables['recommendation_action_matrix'])

## Figures

Figures are exported to `outputs/completed_thesis/figures/` for manuscript insertion.